In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, OneHotEncoder,StandardScaler
from sklearn.metrics import accuracy_score, roc_auc_score, log_loss
from sklearn.svm import SVC
from tqdm import tqdm
from sklearn.compose import ColumnTransformer
from sklearn.compose import make_column_selector

In [2]:
hr=pd.read_csv("D:\\AshleshaRuchika\\PGCP-AI\\Machine Learning\\HR_comma_sep.csv")
hr

,satisfaction_level,last_evaluation,number_project,average_montly_hours,time_spend_company,Work_accident,left,promotion_last_5years,Department,salary
0,0.38,0.53,2,157,3,0,1,0,sales,low
1,0.80,0.86,5,262,6,0,1,0,sales,medium
2,0.10,0.77,6,247,4,0,1,0,sales,low
3,0.92,0.85,5,259,5,0,1,0,sales,low
4,0.89,1.00,5,224,5,0,1,0,sales,low
...,...,...,...,...,...,...,...,...,...,...
14990,0.40,0.57,2,151,3,0,1,0,support,low
14991,0.37,0.48,2,160,3,0,1,0,support,low
14992,0.37,0.53,2,143,3,0,1,0,support,low
14993,0.11,0.96,6,280,4,0,1,0,support,low


In [3]:
le=LabelEncoder()
hr['salary']=le.fit_transform(hr['salary'])
X,y=hr.drop('salary',axis=1), hr['salary']
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.3,random_state=26,stratify=y)

In [4]:
ohe=OneHotEncoder(sparse_output=False,drop="first").set_output(transform="pandas")
transf=ColumnTransformer(transformers=[("OHE",ohe, make_column_selector
                                        (dtype_include=object))],remainder="passthrough",verbose_feature_names_out=False).set_output(transform="pandas")
X_trn_ohe=transf.fit_transform(X_train)
X_tst_ohe=transf.transform(X_test)

In [5]:
svm=SVC(kernel='linear',probability=True)
svm.fit(X_trn_ohe,y_train)
y_pred_prob=svm.predict_proba(X_tst_ohe)
log_loss(y_test,y_pred_prob)

0.9001195549266954

In [6]:
scaler=StandardScaler().set_output(transform="pandas")

In [7]:
X_trn_scl=scaler.fit_transform(X_trn_ohe)#mean and SD
X_tst_scl=scaler.transform(X_tst_ohe)

In [8]:
#linear kernal

In [9]:
Cs=np.linspace(0.001,5,20)
scores=[]
for c in tqdm(Cs):
    svm=SVC(kernel='linear',C=c,probability=True, random_state=26)
    svm.fit(X_trn_scl,y_train)
    y_pred_prob=svm.predict_proba(X_tst_scl)
    scores.append([c,log_loss(y_test,y_pred_prob)])
df_scores=pd.DataFrame(scores,columns=['C','score'])
df_scores.sort_values('score',ascending=True)

100%|██████████████████████████████████████████████████████████████████████████████████| 20/20 [13:29<00:00, 40.47s/it]


,C,score
1,0.264105,0.902746
9,2.368947,0.902746
5,1.316526,0.902746
16,4.210684,0.902746
14,3.684474,0.902747
3,0.790316,0.902747
8,2.105842,0.902747
11,2.895158,0.902747
6,1.579632,0.902747
19,5.000000,0.902747


In [10]:
## polynomial kernel

In [12]:
Cs=np.linspace(0.01,5,20)
Ds=[2,3,4]
scores=[]
for c in Cs:
    for d in Ds:
        svm=SVC(kernel='poly',C=c,probability=True, random_state=26, degree=d)
        svm.fit(X_trn_scl,y_train)
        y_pred_prob=svm.predict_proba(X_trn_scl)
        scores.append([c,d,log_loss(y_test,y_pred_prob)])
df_scores=pd.DataFrame(scores,columns=['C','dergee','score'])
df_scores.sort_values('score',ascending=True)

ValueError: Found input variables with inconsistent numbers of samples: [10496, 4499]

In [ ]:
## Radial Kernel

In [ ]:
Cs=np.linspace(0.001,5,20)
Gs=np.linspace(0.001,5,20)
Ds=[2,3,4]
scores=[]
for c in Cs:
    for g in Gs:
        svm=SVC(kernel='rbf',C=c,probability=True, random_state=26,gamma=g)
        svm.fit(X_train,y_train)
        y_pred_prob=svm.predict_proba(X_test)
        scores.append([c,d,log_loss(y_test,y_pred_prob)])
df_scores=pd.DataFrame(scores,columns=['C','gamma','score'])
df_scores.sort_values('score',ascending=True)

In [ ]:
## sigmoid Kernal

In [ ]:
Cs=np.linspace(0.001,5,20)
coef=np.linspace(-5,5,20)
Ds=[2,3,4]
scores=[]
for c in Cs:
    for cf in coef:
        svm=SVC(kernel='sigmoid',C=c,probability=True, random_state=26,coef0=cf)
        svm.fit(X_train,y_train)
        y_pred_prob=svm.predict_proba(X_test)
        scores.append([c,d,log_loss(y_test,y_pred_prob)])
df_scores=pd.DataFrame(scores,columns=['C','coef0','score'])
df_scores.sort_values('score',ascending=True)